In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_athens = pd.read_csv('athens.csv')
df_athens

In [ ]:
df_athens.dtypes

In [ ]:
df_athens['Date'] = pd.to_datetime(df_athens['Date'])

In [ ]:
df_athens.columns

In [ ]:
len(df_athens.columns)

In [ ]:
df_athens[df_athens.isnull().any(axis=1)].reset_index()

In [ ]:
(df_athens.isnull().sum() / len(df_athens) * 100).sort_values(ascending=False)

In [ ]:
df_athens["station_name"].value_counts()

In [ ]:
df_athens["code"].isna().sum()

In [ ]:
df_athens['code'] = df_athens['code'].fillna('ANO001')

In [ ]:
(df_athens.isnull().sum() / len(df_athens) * 100).sort_values(ascending=False)

In [ ]:
cols = [
    "Vegitation (High)", "Wind-Speed (U)", "Wind-Speed (V)", 
    "Dewpoint Temp", "Soil Temp", "Total Percipitation", 
    "Vegitation (Low)", "Relative Humidity"
]

In [ ]:
same_missing_pattern = df_athens[cols].isna().eq(df_athens[cols[0]].isna(), axis=0).all().all()

print("Do all these columns have the same missing pattern?", same_missing_pattern)

In [ ]:
df_athens.iloc[[463892]]

In [ ]:
cols_to_fill = [
    "Vegitation (High)", "Wind-Speed (U)", "Wind-Speed (V)", 
    "Dewpoint Temp", "Soil Temp", "Total Percipitation", 
    "Vegitation (Low)", "Relative Humidity"
]

original_index = df_athens.index
original_order = df_athens.copy()

df_athens = df_athens.sort_values(['station_name', 'Date'])
df_athens[cols_to_fill] = df_athens.groupby('station_name')[cols_to_fill].fillna(method='ffill')
df_athens = df_athens.reindex(original_index)

In [ ]:
df_athens.iloc[[463850]]

In [ ]:
(df_athens.isnull().sum() / len(df_athens) * 100).sort_values(ascending=False)

In [ ]:
original_index = df_athens.index

df_athens_sorted = df_athens.sort_values(['station_name', 'Date']).copy()

df_athens_sorted['Temp'] = (
    df_athens_sorted
    .groupby('station_name')['Temp']
    .transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
)

df_athens = df_athens_sorted.reindex(original_index)

In [ ]:
(df_athens.isnull().sum() / len(df_athens) * 100).sort_values(ascending=False)

In [ ]:
stations_under_10300_O3 = (
    df_athens[df_athens['O3'].isna()]
    ['station_name']
    .value_counts()
    .loc[lambda x: x < 10300]
    .index
)

stations_under_10300_NO2 = (
    df_athens[df_athens['NO2'].isna()]
    ['station_name']
    .value_counts()
    .loc[lambda x: x < 10300]
    .index
)

df_athens = df_athens.sort_values(['station_name', 'Date'])
df_athens.loc[
    df_athens['station_name'].isin(stations_under_10300_O3), 'O3'
] = (
    df_athens[df_athens['station_name'].isin(stations_under_10300_O3)]
    .groupby('station_name')['O3']
    .fillna(method='ffill')
)

df_athens.loc[
    df_athens['station_name'].isin(stations_under_10300_NO2), 'NO2'
] = (
    df_athens[df_athens['station_name'].isin(stations_under_10300_NO2)]
    .groupby('station_name')['NO2']
    .fillna(method='ffill')
)

df_athens = df_athens.reindex(original_index)



In [ ]:
(df_athens.isnull().sum() / len(df_athens) * 100).sort_values(ascending=False)

In [ ]:
df_athens[df_athens['O3'].isna()]["station_name"].value_counts()

In [ ]:
df_athens.loc[df_athens['station_name'] == 'ATHINAS', 'NO2'] = (
    df_athens.loc[df_athens['station_name'] == 'ATHINAS', 'NO2'].fillna(35.0)
)
df_athens.loc[df_athens['station_name'] == 'ATHINAS', 'O3'] = (
    df_athens.loc[df_athens['station_name'] == 'ATHINAS', 'O3'].fillna(2.0)
)

In [ ]:
df_athens[df_athens['O3'].isna()]["station_name"].value_counts()

In [ ]:
df_pivot = df_athens.pivot_table(
    index='Date',
    columns='station_name',
    values='PM2.5'
)
df_pivot

In [ ]:
corr = df_pivot.corr()
corr.index.name = None
corr

In [ ]:
results = []

corr_matrix = corr.copy()

for column in corr_matrix.columns:
    top_2 = corr_matrix[column].nlargest(2)
    second_corr = top_2.iloc[1]
    second_station = top_2.index[1]
    
    results.append({
        'Station': column,
        'Second_Most_Correlated_With': second_station,
        'Correlation_Value': second_corr,
        'Correlation_Percentage': f"{second_corr*100:.1f}%"
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df[results_df["Station"] == "ATHENS_01"]

In [ ]:
results_df[results_df["Station"] == "ATHENS_02"]

In [ ]:
results_df[results_df["Station"] == "ATH-ENVICARE-3"]

In [ ]:
station_mapping = {
    'ATHENS_01': 'PANACEA_AirPaP_001', 
    'ATHENS_02': 'National Technical University of Athens',
    'ATH-ENVICARE-3': 'PANACEA_029',
    'CleanAir in Greece - Thiseio': 'PPCITY21',
    'CleanAir in Greece - Thiseio II': 'PPCITY21',
}

cols_to_copy = ["O3", "NO2"]

for target, source in station_mapping.items():
    for col in cols_to_copy:
        df_athens.loc[df_athens["station_name"] == target, col] = (
            df_athens.loc[df_athens["station_name"] == source, col].values
        )

In [ ]:
df_athens

In [ ]:
(df_athens.isnull().sum() / len(df_athens) * 100).sort_values(ascending=False)

In [ ]:
df_athens['Wind_Speed'] = np.sqrt(df_athens['Wind-Speed (U)']**2 + df_athens['Wind-Speed (V)']**2)
df_athens

In [ ]:
df_athens['Point'] = df_athens.apply(lambda row: (row['Latitude'], row['Longitude']), axis=1)

df_athens = df_athens.drop(columns=['Latitude', 'Longitude'])

df_athens

In [ ]:
pm25_edges = [0, 12.0, 35.4, 55.4, 150.4, 250.4, np.inf]
pm25_labels = ["Good", "Moderate", "Unhealthy for SG", "Unhealthy", "Very Unhealthy", "Hazardous"]

In [ ]:
pm10_edges = [0, 54, 154, 254, 354, 424, np.inf]
pm10_labels = ["Good", "Moderate", "Unhealthy for SG", "Unhealthy", "Very Unhealthy", "Hazardous"]

In [ ]:
def add_pm_bins(df, col, edges, labels, out_col):
    if col in df.columns:
        df[out_col] = pd.cut(df[col], bins=edges, labels=labels, include_lowest=True)
        print(f"Created {out_col} from {col}")
    else:
        print(f"Skipped: column '{col}' not found.")

In [ ]:
add_pm_bins(df_athens, 'PM2.5', pm25_edges, pm25_labels, 'PM25_AQI_bin')
add_pm_bins(df_athens, 'PM10', pm10_edges, pm10_labels, 'PM10_AQI_bin')


In [ ]:
df_athens

In [ ]:
df_athens["PM10_AQI_bin"].value_counts()

In [ ]:
df_athens["PM25_AQI_bin"].value_counts()

In [ ]:
df_athens['Temperature_Binary'] = (df_athens['Temp'] > 15).astype(int)
df_athens


In [ ]:
df_athens["Temperature_Binary"].value_counts()

In [ ]:
df_athens.to_csv('athens_results.csv')

In [ ]:
from statsmodels.tsa.seasonal import STL

def robust_zscore(x):
    x = np.asarray(x)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad == 0:
        std = np.std(x)
        return (x - med) / std if std != 0 else np.zeros_like(x)
    return (x - med) / (1.4826 * mad)



In [ ]:
def safe_name(colname):
    return str(colname).replace('.', '_').replace(' ', '_').replace('-', '_')



In [ ]:
def compute_zresids_stl(df,
                        dt_col='Date',
                        station_col='station_name',
                        vars_to_process=None,
                        period=24):

    if vars_to_process is None:
        vars_to_process = ['PM10', 'PM2.5', 'NO2', 'O3', 'Temp']

    df = df.copy()
    df[dt_col] = pd.to_datetime(df[dt_col])
    df = df.sort_values([station_col, dt_col])

    zframes = []

    for var in vars_to_process:
        if var not in df.columns:
            raise KeyError(f"Column '{var}' not found in dataframe")

        results = []
        for station, g in df.groupby(station_col, sort=False):
            s = g.set_index(dt_col)[var].astype(float)

            if len(s) < period * 2:
                # not enough data -> produce NaNs for zresid
                out = pd.DataFrame({'z_resid': [np.nan] * len(s)}, index=s.index)
            else:
                stl = STL(s, period=period, robust=True)
                res = stl.fit()
                resid = res.resid.values
                z = robust_zscore(resid)
                out = pd.DataFrame({'z_resid': z}, index=s.index)

            out = out.reset_index().rename(columns={dt_col: 'dt_index'})
            out[station_col] = station
            results.append(out)

        all_out = pd.concat(results, ignore_index=True).rename(columns={'dt_index': dt_col})
        colname = f"{safe_name(var)}_z_resid"
        all_out = all_out[[dt_col, station_col, 'z_resid']].rename(columns={'z_resid': colname})

        zframes.append(all_out)

    merged_z = None
    for zf in zframes:
        if merged_z is None:
            merged_z = zf
        else:
            merged_z = pd.merge(merged_z, zf, on=[dt_col, station_col], how='outer')

    df_out = pd.merge(df, merged_z, on=[dt_col, station_col], how='left')

    return df_out


In [ ]:
def apply_thresholds(df_with_z,
                     vars_to_process=None,
                     z_thresh=None,
                     dt_col='Date'):

    df = df_with_z.copy()
    if vars_to_process is None:
        zcols = [c for c in df.columns if c.endswith('_z_resid')]
        vars_to_process = [c[:-7].replace('_', '.') for c in zcols]

    if isinstance(z_thresh, dict):
        thresh_map = z_thresh
    else:
        thresh_map = {v: z_thresh for v in vars_to_process}

    for var in vars_to_process:
        suf = safe_name(var)
        zcol = f"{suf}_z_resid"
        if zcol not in df.columns:
            raise KeyError(f"Expected z-resid column '{zcol}' not found. Run compute_zresids_stl first.")

        t = thresh_map.get(var, thresh_map.get(suf, None))
        if t is None:
            raise ValueError(f"No threshold specified for variable '{var}' (provide z_thresh numeric or dict).")

        two_col = f"{suf}_outlier_two_sided"
        pos_col = f"{suf}_outlier_positive_spike"

        df[two_col] = df[zcol].abs() > t
        df[pos_col] = df[zcol] > t

        df[two_col] = df[two_col].fillna(False)
        df[pos_col] = df[pos_col].fillna(False)

    return df

In [ ]:
vars_to_do = ['PM10', 'PM2.5', 'NO2', 'O3']

df_z = compute_zresids_stl(
    df_athens,
    dt_col='Date',
    station_col='station_name',
    vars_to_process=vars_to_do,
    period=24
)


per_var_thresh = {
    'PM10': 14.0,
    'PM2.5': 12.0,
    'NO2': 10.0,
    'O3': 8.0
}

df_final = apply_thresholds(
    df_z,
    vars_to_process=vars_to_do,
    z_thresh=per_var_thresh
)

df_final

In [ ]:
import matplotlib.pyplot as plt

def plot_station_outliers(station_name, df_final):
    station_df = df_final[df_final['station_name'] == station_name].sort_values('Date')
    
    fig, axes = plt.subplots(4, 3, figsize=(15, 12))
    axes = axes.flatten()
    
    for idx, var in enumerate(['PM10', 'PM2.5', 'NO2', 'O3']):
        suf = safe_name(var)
        zcol = f"{suf}_z_resid"
        outlier_col = f"{suf}_outlier_two_sided"
        
        # Time series with outliers highlighted
        axes[idx*3].plot(station_df['Date'], station_df[var], alpha=0.5, label='Original')
        outliers = station_df[station_df[outlier_col]]
        axes[idx*3].scatter(outliers['Date'], outliers[var], 
                          color='red', s=10, label='Outlier')
        axes[idx*3].set_title(f'{var} - {station_name}')
        axes[idx*3].legend()
        
        # Z-residuals
        axes[idx*3+1].plot(station_df['Date'], station_df[zcol], alpha=0.5)
        axes[idx*3+1].axhline(y=per_var_thresh[var], color='r', linestyle='--', label=f'Threshold: {per_var_thresh[var]}')
        axes[idx*3+1].axhline(y=-per_var_thresh[var], color='r', linestyle='--')
        axes[idx*3+1].set_title(f'{var} Z-Residuals')
        
        # Histogram of z-residuals
        axes[idx*3+2].hist(station_df[zcol].dropna(), bins=50, edgecolor='black')
        axes[idx*3+2].axvline(x=per_var_thresh[var], color='r', linestyle='--')
        axes[idx*3+2].axvline(x=-per_var_thresh[var], color='r', linestyle='--')
        axes[idx*3+2].set_title(f'{var} Z-Residual Distribution')
    
    plt.tight_layout()
    plt.show()

# Check the problematic station
plot_station_outliers('ARISTOTELOUS', df_final)